# Travis County Week 1 EDA

> Research lead: Khadeja

This notebook is the week 1 exploration workspace for the Travis County criminal justice dataset. The immediate goals are to:

- inventory the available files
- preview the core CSV schemas
- identify likely linkage keys across person, booking, case, and pretrial records
- flag proxy-variable risks and open data questions for Friday discussion

In [11]:
import csv
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [12]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CORE_FILES = [
    "Booking2010_2012v3.csv",
    "Booking2013_2016v3.csv",
    "CaseData_v2.csv",
    "PreTrial Charge-Interview.csv",
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory exists: {RAW_DATA_DIR.exists()}")
print(f"Pandas version: {pd.__version__}")

Project root: C:\Users\khade\Documents\GitHub\Bias-and-Fairness-in-AI-governance
Raw data directory exists: True
Pandas version: 3.0.3


In [13]:
def build_inventory(data_dir: Path) -> list[dict[str, object]]:
    records = []
    for path in sorted(data_dir.iterdir()):
        if path.is_file():
            records.append(
                {
                    "file": path.name,
                    "suffix": path.suffix.lower(),
                    "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
                }
            )
    return records

inventory = build_inventory(RAW_DATA_DIR)
inventory_df = pd.DataFrame(inventory).sort_values(["suffix", "file"]).reset_index(drop=True)

print(f"Raw file count: {len(inventory_df)}")
display(inventory_df)

Raw file count: 18


,file,suffix,size_mb
0,Booking2010_2012v3.csv,.csv,62.78
1,Booking2013_2016v3.csv,.csv,63.32
2,CaseData_v2.csv,.csv,110.94
3,PreTrial Charge-Interview.csv,.csv,83.48
4,chargecode_description.dta,.dta,0.43
5,171017_CCHCodes_Violent marked by Slayton.xlsx,.xlsx,0.11
6,CaseIDtoBookingChargeIDUPDATED.xlsx,.xlsx,8.83
7,Events.xlsx,.xlsx,4.12
8,Mental Health Flag Events.xlsx,.xlsx,3.69
9,PersonData.xlsx,.xlsx,5.21


In [14]:
def read_csv_header(path: Path) -> list[str]:
    with path.open("r", newline="", encoding="utf-8-sig") as handle:
        reader = csv.reader(handle)
        return next(reader)

def preview_rows(path: Path, limit: int = 3) -> list[dict[str, str]]:
    rows = []
    with path.open("r", newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        for index, row in enumerate(reader):
            if index >= limit:
                break
            rows.append(row)
    return rows

schema_preview = []
for file_name in CORE_FILES:
    path = RAW_DATA_DIR / file_name
    header = read_csv_header(path)
    schema_preview.append(
        {
            "file": file_name,
            "column_count": len(header),
            "first_columns": ", ".join(header[:12]),
        }
    )

schema_preview_df = pd.DataFrame(schema_preview)
display(schema_preview_df)

,file,column_count,first_columns
0,Booking2010_2012v3.csv,43,"bookingID, BookingNumber, mni, PersonID, jailID, CustodyStatusID, CustodyTypeID, bookingdate, bookingtime, ReleaseDa..."
1,Booking2013_2016v3.csv,43,"bookingID, BookingNumber, mni, PersonID, jailID, CustodyStatusID, CustodyTypeID, bookingdate, bookingtime, ReleaseDa..."
2,CaseData_v2.csv,81,"CaseID, DivisionID, CourtTypeID, CaseTypeID, CaseStatusID, StatusDate, CaseStateID, SealedFlag, RestrictedFlag, Case..."
3,PreTrial Charge-Interview.csv,37,"PA_MAST_NO, PK_BKG_NO, BJ_BK_DATE, PK_CHG_DT, INT_OFF, OFF_REC, PK_BND_REC, PK_BND_GRT, PK_GRNT_DT, PK_ATY_REC, PK_A..."


In [15]:
sample_file = RAW_DATA_DIR / "CaseData_v2.csv"
sample_rows = preview_rows(sample_file, limit=5)
sample_df = pd.DataFrame(sample_rows)

display(sample_df)

,CaseID,DivisionID,CourtTypeID,CaseTypeID,CaseStatusID,StatusDate,CaseStateID,SealedFlag,RestrictedFlag,CaseInitiationDate,DateCaseEntered,ReindictRefile,CaseStateIssue,LatestCasePhaseID,LatestAgeofPendingComplaint,CriminalDefendantID,PartyID,PersonID,MNI,OriginalBookingNumber,BookingDate,IndictFilingDate,CalendarDaysBookingtoIndictFiling,JailCaseTypeID,HighestLevelChargeCount,HighestLevelDispositionCount,CaseChargeID,CountNumber,ChargeCode,CsMgmtChargeID,CurrentChargeStateID,ChargeReducedEnhanced,OriginalDispositionPleaID,OriginalDispositionPleaDate,OriginalDispositionTrialTypeID,OriginalDispositionVerdictID,OriginalDispositionVerdictDate,OriginalDispositionMethodID,OriginalDispositionID,OriginalDispositionDate,OriginalDispositionEvent,OriginalSentenceID,LatestDispositionMethodID,LatestDispositionID,LatestDispositionDate,LatestSentenceID,EarliestKnownChargeID,OffenseDate,CriminalDefendantHistoryID,ActivationDate,ActivationEventID,ActivationTypeID,AssociatedBookingNumber,AssociatedBookingDate,InactivityDays,CalendarDaysFilingtoCurrent,AgeofCase,DispositionID,DispositionDate,DispositionEventID,DispositionTypeID,CalendarDaysFilingtoDisposition,TimetoDisposition,casephaseID,CasePhaseCode,UnabletoCalcAgeInactivity,DispositionMethodID,PartyAttorneyID,PartyIDRepresenting,attorneyID,AttorneySBN,AttorneyCounselTypeID,AttorneyCaseStatusID,AttorneyStatusDateTime,CurrentAttorneyFlag,AttorneyStartDate,AttorneyEndDate,CurrentLeadAttorneyFlag,AssignedCourtID,OriginalDispositionCourtID,LatestDispositonCourtID
0,1381881,2,1,203,142,11/18/2010,3,0,0,8/30/2010,8/30/2010,NULL,NULL,NULL,NULL,911820,3383654,250381,847199,1041441,8/26/2010,9/3/2010,9,4,1,1,18919,1,41990018,5475,2,NULL,2,11/18/2010,2,NULL,NULL,2,171,11/18/2010,9038,20,NULL,NULL,NULL,NULL,5475,8/26/2010,1855380,9/3/2010,5559,31,1041441,8/26/2010,70,NULL,NULL,171,11/18/2010,4842,1,77,7,1,ORIGINAL,0,2,731706,3383654,973,784356,17,8,11/16/2010,Y,NULL,NULL,Y,J65,J65,NULL
1,1398852,2,1,195,142,10/2/2013,3,0,0,1/5/2011,1/5/2011,NULL,NULL,NULL,NULL,445098,3417539,305159,1125250,1062013,12/28/2010,2/2/2011,37,5,1,1,18920,1,35620008,1716,2,NULL,2,9/14/2011,2,NULL,NULL,2,171,9/14/2011,9038,20,NULL,NULL,NULL,NULL,1716,12/3/2010,1485448,2/2/2011,5559,31,1062013,12/28/2010,3,NULL,NULL,171,9/14/2011,4842,1,225,222,1,ORIGINAL,0,2,513670,3417539,53004,13142050,30,8,2/8/2011,Y,NULL,NULL,Y,J65,J65,NULL
2,1437533,2,1,203,142,10/2/2012,3,0,0,11/21/2011,11/21/2011,NULL,NULL,NULL,NULL,171165,3497488,420243,1811100,1153225,11/19/2011,12/19/2011,31,5,1,1,18939,1,54040009,1513,2,NULL,2,10/2/2012,2,NULL,NULL,2,171,10/2/2012,9038,20,NULL,NULL,NULL,NULL,1513,11/19/2011,1499896,12/19/2011,5559,31,1153225,11/19/2011,NULL,NULL,NULL,171,10/2/2012,4842,1,289,289,1,ORIGINAL,0,2,566513,3497488,98561,24038749,30,8,12/16/2011,Y,NULL,NULL,Y,J33,J33,NULL
3,1513292,2,1,203,142,10/2/2015,3,0,0,7/11/2013,7/11/2013,NULL,NULL,NULL,NULL,921091,3644210,441243,1908301,1329693,7/10/2013,8/6/2013,28,5,1,1,18943,1,54040009,1513,2,NULL,1,12/2/2013,2,NULL,NULL,2,101,12/2/2013,9002,20,4,102,10/2/2015,20,1513,7/10/2013,1901930,8/6/2013,5559,31,1329693,7/10/2013,NULL,NULL,NULL,101,12/2/2013,4806,1,119,119,1,ORIGINAL,0,2,1210246,3644210,9000,792684,17,8,9/30/2015,Y,NULL,NULL,Y,J33,J33,J10
4,1506253,2,1,195,138,9/24/2013,3,0,0,5/24/2013,5/24/2013,NULL,NULL,NULL,NULL,486104,3633626,246217,796986,1321738,5/21/2013,6/14/2013,25,5,1,1,18956,1,54990067,4148,2,NULL,NULL,NULL,2,NULL,NULL,2,147,9/24/2013,9030,20,NULL,NULL,NULL,NULL,4148,4/30/2013,1904033,6/14/2013,5559,31,1321738,5/21/2013,NULL,NULL,NULL,147,9/24/2013,4834,4,103,103,1,ORIGINAL,0,2,754205,3633626,112459,24058588,24,8,6/19/2013,Y,NULL,NULL,Y,J33,J33,NULL


In [16]:
linkage_hypotheses = [
    {"entity": "person", "candidate_keys": "PersonID, mni/MNI", "source_files": "Booking, Case, likely PersonData"},
    {"entity": "booking", "candidate_keys": "bookingID, BookingNumber, AssociatedBookingNumber, PK_BKG_NO", "source_files": "Booking, Case, PreTrial"},
    {"entity": "charge", "candidate_keys": "BookingChargeID, CaseChargeID, ChargeCode", "source_files": "Booking, Case, charge lookup"},
    {"entity": "case", "candidate_keys": "CaseID", "source_files": "CaseData, bridge workbook"},
    {"entity": "pretrial interview", "candidate_keys": "PA_MAST_NO, PK_BKG_NO, PK_CHARGE", "source_files": "PreTrial Charge-Interview"},
]

linkage_df = pd.DataFrame(linkage_hypotheses)
display(linkage_df)

,entity,candidate_keys,source_files
0,person,"PersonID, mni/MNI","Booking, Case, likely PersonData"
1,booking,"bookingID, BookingNumber, AssociatedBookingNumber, PK_BKG_NO","Booking, Case, PreTrial"
2,charge,"BookingChargeID, CaseChargeID, ChargeCode","Booking, Case, charge lookup"
3,case,CaseID,"CaseData, bridge workbook"
4,pretrial interview,"PA_MAST_NO, PK_BKG_NO, PK_CHARGE",PreTrial Charge-Interview


## Priority Workbook Inspection

This section starts the week 1 inspection of the three highest-priority Excel workbooks:

- `PersonData.xlsx`
- `PreTrial to Booking Info.xlsx`
- `CaseIDtoBookingChargeIDUPDATED.xlsx`

The goal is to identify sheet structure, likely join keys, and the first columns worth documenting in the data inventory.

In [17]:
PRIORITY_WORKBOOKS = [
    "PersonData.xlsx",
    "PreTrial to Booking Info.xlsx",
    "CaseIDtoBookingChargeIDUPDATED.xlsx",
]

def inspect_workbook(file_name: str) -> list[dict[str, object]]:
    workbook_path = RAW_DATA_DIR / file_name
    excel_file = pd.ExcelFile(workbook_path)
    rows = []
    for sheet_name in excel_file.sheet_names:
        sheet_preview = pd.read_excel(workbook_path, sheet_name=sheet_name, nrows=0)
        rows.append(
            {
                "workbook": file_name,
                "sheet_name": sheet_name,
                "column_count": len(sheet_preview.columns),
                "first_columns": ", ".join(sheet_preview.columns.astype(str)[:10]),
            }
        )
    return rows

workbook_sheet_rows = []
for workbook_name in PRIORITY_WORKBOOKS:
    workbook_sheet_rows.extend(inspect_workbook(workbook_name))

workbook_sheet_df = pd.DataFrame(workbook_sheet_rows)
display(workbook_sheet_df)

,workbook,sheet_name,column_count,first_columns
0,PersonData.xlsx,PersonData,7,"PersonID, MNI, DateofBirth, GenderID, RaceID, EthnicityID, CitizenshipID"
1,PreTrial to Booking Info.xlsx,Booking,11,"PA_MAST_NO, BH_BKG_NO, BJ_BK_DATE, BH_WAR_NO, BH_C_CASE, BH_BAIL_AMT, BH_CHG_DATE, BH_JCHG_NO, BH_CRT_DATE, BH_CRT_TIME"
2,CaseIDtoBookingChargeIDUPDATED.xlsx,Sheet2,2,"BookingChargeID, CaseID"
3,CaseIDtoBookingChargeIDUPDATED.xlsx,Sheet3,0,


In [18]:
for workbook_name in PRIORITY_WORKBOOKS:
    workbook_path = RAW_DATA_DIR / workbook_name
    excel_file = pd.ExcelFile(workbook_path)
    first_sheet_name = excel_file.sheet_names[0]
    preview_df = pd.read_excel(workbook_path, sheet_name=first_sheet_name, nrows=5)

    print(f"Workbook: {workbook_name}")
    print(f"First sheet: {first_sheet_name}")
    display(preview_df.head())

Workbook: PersonData.xlsx
First sheet: PersonData


,PersonID,MNI,DateofBirth,GenderID,RaceID,EthnicityID,CitizenshipID
0,69724,1172445,1984-06-19,1,7,NaN,1
1,135247,782761,1974-09-15,1,7,1.0,1
2,135253,782777,1977-02-22,2,7,NaN,1
3,135265,782803,1977-08-16,1,7,1.0,2
4,135273,782741,1982-04-16,1,7,1.0,1


Workbook: PreTrial to Booking Info.xlsx
First sheet: Booking


,PA_MAST_NO,BH_BKG_NO,BJ_BK_DATE,BH_WAR_NO,BH_C_CASE,BH_BAIL_AMT,BH_CHG_DATE,BH_JCHG_NO,BH_CRT_DATE,BH_CRT_TIME,BH_CRT_EVT
0,PT10000002,1000003,2010-01-01,NaN,C1CR10200039,5000,2010-01-01,1,2010-01-01,816,ARR
1,PT10000003,1000003,2010-01-01,NaN,C1CR10200039,5000,2010-01-01,1,2010-01-01,816,ARR
2,PT10000005,1000008,2010-01-01,NaN,NaN,0,2010-01-01,2,NaT,0,NaN
3,PT10000005,1000008,2010-01-01,NaN,NaN,0,2010-01-01,1,NaT,0,NaN
4,PT10000005,1000008,2010-01-01,NaN,C1CR10200038,1000,2010-01-01,3,2010-01-01,816,ARR


Workbook: CaseIDtoBookingChargeIDUPDATED.xlsx
First sheet: Sheet2


,BookingChargeID,CaseID
0,1276847,408393
1,1265925,408402
2,1512394,408402
3,1544613,408402
4,1440905,408402


## Initial Inspection Findings

- `PersonData.xlsx` appears to contain direct demographic attributes at the person level, including `GenderID`, `RaceID`, `EthnicityID`, and `CitizenshipID`. This is the clearest current source for protected or sensitive attributes.
- `PreTrial to Booking Info.xlsx` looks like a bridge between pretrial records and booking or case records because it contains `PA_MAST_NO`, booking-number style fields, case-like fields, charge dates, and court event fields.
- `CaseIDtoBookingChargeIDUPDATED.xlsx` is a narrow mapping table that directly links `BookingChargeID` to `CaseID`, which should help connect booking charge records to case-level outcomes.
- These three workbooks materially strengthen the current linkage hypothesis across person, booking, charge, case, and pretrial entities.

## Week 1 Questions To Resolve

1. Which workbook contains the clearest demographic variables for protected-attribute analysis?
2. How exactly do `PK_BKG_NO`, `BookingNumber`, and `AssociatedBookingNumber` line up?
3. Which files are true event logs versus lookup tables or bridge tables?
4. Which variables are operational decision outputs that should not be treated as neutral predictors?

Use the inventory and schema previews above to guide the next pass through the Excel workbooks.